# Session 2 — TPU: Sequence Length Scaling

## Background

Attention's memory footprint grows as O(seq²): the attention matrix is `[batch, heads, seq_len, seq_len]` and must be materialized and transposed at every step. Session 1 showed that the GPU's memory bus (~300 GB/s) becomes the bottleneck as batch size grows. Sequence length applies the same pressure through a different dimension, but with a quadratic rather than linear relationship. The TPU's HBM2 bus (~819 GB/s) provides 2.7× more bandwidth — if this is the binding constraint, the gap should widen as sequences grow.

## Goal

At the end of this session, participants will be able to explain how attention's quadratic memory footprint interacts differently with each device's memory architecture, will have measured how the throughput ratio shifts as sequences lengthen, and will know which device to reach for when their research involves long-context models.

## Question

Does the TPU's memory bandwidth advantage compound as sequences grow, or does the crossover point move?

---

**Runtime:** TPU (v5litepod-1 or equivalent)

After running, results are saved to `results/tpu_seq_sweep.json` and loaded automatically by `03-analysis.ipynb`.

**Note:** The first step at each seq_len triggers XLA recompilation (new graph shape). This is expected and is absorbed by the 5-step warmup.

In [ ]:
import subprocess, sys, time
import torch, torch.nn as nn

try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.runtime as xr
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "torch==2.9.0", "torch_xla[tpu]==2.9.0",
        "-f", "https://storage.googleapis.com/libtpu-releases/index.html"
    ])
    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.runtime as xr

# ── TPU info ──────────────────────────────────────────────────────────────────
device = torch_xla.device()

try:
    from torch_xla._internal import tpu as _tpu
    tpu_env  = _tpu.get_tpu_env()
    chip     = tpu_env.get("ACCELERATOR_TYPE") or tpu_env.get("TPU_ACCELERATOR_TYPE", "unknown")
    n_chips  = xr.global_runtime_device_count()
except Exception:
    chip, n_chips = "unknown", "unknown"

print(f"Python    : {sys.version.split()[0]}")
print(f"PyTorch   : {torch.__version__}")
print(f"torch_xla : {torch_xla.__version__}")
print(f"Device    : {device}")
print(f"TPU chip  : {chip}")
print(f"N chips   : {n_chips}")

In [ ]:
import sys; sys.path.insert(0, "..")
from transformer_block import BenchmarkTransformerBlock, D_MODEL, N_HEAD, DIM_FEEDFORWARD

# ── Session 2 config ──────────────────────────────────────────────────────────
BATCH_SIZE  = 32   # fixed at Session 1 crossover point
SEQ_LENGTHS = [64, 128, 256, 512, 1024, 2048]
N_STEPS, WARMUP = 50, 5

# ── Benchmark function ────────────────────────────────────────────────────────
def benchmark(seq_len):
    model     = BenchmarkTransformerBlock(d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=DIM_FEEDFORWARD).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss()
    model.train()
    elapsed = 0.0
    for step in range(N_STEPS + WARMUP):
        x = torch.randn(BATCH_SIZE, seq_len, D_MODEL, device=device)
        y = torch.randn(BATCH_SIZE, seq_len, D_MODEL, device=device)
        torch_xla.sync()
        t0 = time.perf_counter()
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        torch_xla.sync()
        if step >= WARMUP:
            elapsed += time.perf_counter() - t0
    return (N_STEPS * BATCH_SIZE) / elapsed

print(f"Config  : batch={BATCH_SIZE}, d_model={D_MODEL}, n_head={N_HEAD}, dim_ff={DIM_FEEDFORWARD}")
print(f"Sweep   : seq_len = {SEQ_LENGTHS}")
print("Model and benchmark function defined.")

In [ ]:
results = {}

for seq_len in SEQ_LENGTHS:
    try:
        print(f"  [TPU] seq_len={seq_len:>5} ... ", end="", flush=True)
        tput = benchmark(seq_len)
        results[seq_len] = round(tput, 1)
        print(f"{tput:,.0f} samples/sec")
    except RuntimeError as e:
        if "out of memory" in str(e).lower() or "resource" in str(e).lower():
            print("OOM — stopping.")
            break
        raise

print("\nDone.")
print("\n── Copy this into 03-analysis.ipynb ──────────────────")
print(f"TPU_RESULTS = {results}")

In [ ]:
import json, os
from datetime import datetime, timezone

def save_results(hardware, results, device_name="", fixed_batch_size=None, results_dir="results"):
    os.makedirs(results_dir, exist_ok=True)
    payload = {
        "hardware":         hardware,
        "device_name":      device_name,
        "session":          "2",
        "axis":             "seq_len",
        "fixed_batch_size": fixed_batch_size,
        "timestamp":        datetime.now(timezone.utc).isoformat(),
        "results":          {str(k): v for k, v in sorted(results.items())},
    }
    path = os.path.join(results_dir, f"{hardware.lower()}_seq_sweep.json")
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)
    return path

path = save_results(
    "TPU", results,
    device_name=chip,
    fixed_batch_size=BATCH_SIZE,
    results_dir="results",
)
print(f"Saved → {path}")